# TTRL + ACE Co-Evolution on AIME 2024
Experiment: Does interleaved co-evolution of model weights (TTRL/GRPO) and strategy memory (ACE playbook) produce synergistic improvement? Extends CG-TTRL (static context + TTRL) to **evolving** context + TTRL with confidence-weighted rewards.

**4 Conditions:**
1. Baseline — frozen model, single generation, ground-truth check
2. ACE-only — frozen model, playbook evolution, majority vote
3. TTRL-only — GRPO weight updates, majority vote, no playbook
4. ACE+TTRL — co-evolution of both weights and playbook

In [ ]:
# Install dependencies (Colab)
# All versions pinned for reproducibility. Tested on Colab A100 40GB.
# NOTE: vLLM >= 0.11.x has a 2.4x perf regression with TRL GRPOTrainer
# (https://github.com/huggingface/trl/issues/4897). If runtime is too slow,
# try downgrading to vllm==0.10.2 (only version with full perf).
# vLLM requires transformers < 5 (https://github.com/vllm-project/vllm/issues/30466).

!pip install trl==0.27.2 vllm==0.12.0 transformers==4.57.3 peft==0.18.1 accelerate==1.12.0 datasets torch==2.6.0 scipy==1.15.2 matplotlib==3.10.0 nest_asyncio==1.6.0 flash-attn==2.7.4 --no-build-isolation

In [ ]:
import copy
import csv
import gc
import json
import os
import re
import time
from abc import ABC, abstractmethod
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# GPU Optimizations (must be set before any CUDA operations)
# ---------------------------------------------------------------------------

# TF32: 8x faster matmul on A100 (Ampere+) with no accuracy loss for bf16 training
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Prevent CUDA memory fragmentation (expandable_segments avoids OOM from
# fragmented reserved memory; max_split_size_mb reduces splitting overhead)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"

# vLLM uses asyncio internally; Jupyter/Colab already has a running event loop.
# Without nest_asyncio, vLLM LLM() instantiation crashes with:
# "RuntimeError: This event loop is already running"
import nest_asyncio
nest_asyncio.apply()

# ---------------------------------------------------------------------------
# GPU Detection + Adaptive Config
# ---------------------------------------------------------------------------

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)")
else:
    gpu_name = "CPU"
    gpu_mem_gb = 0
    print("WARNING: No GPU detected. This notebook requires a GPU.")


@dataclass
class Config:
    """All hyperparameters for the TTRL+ACE experiment."""
    # Model
    MODEL_NAME: str = "Qwen/Qwen2.5-Math-7B-Instruct"

    # Generation
    NUM_GENERATIONS: int = 16
    MAX_BULLETS: int = 20
    MAX_COMPLETION_LENGTH: int = 3072    # TTRL paper uses 3072 for AIME

    # GRPO / RL
    KL_COEFF: float = 0.0
    LORA_RANK: int = 64                  # RL needs rank >= 32 (veRL docs); 64 for math
    LORA_ALPHA: int = 64                 # 1:1 ratio at higher ranks
    LORA_MODULES: List[str] = field(     # Attention + MLP (gate_proj critical for math)
        default_factory=lambda: [
            "q_proj", "k_proj", "v_proj", "o_proj",   # Attention
            "gate_proj", "up_proj", "down_proj",       # MLP
        ]
    )
    GRPO_EPOCHS: int = 20               # Training epochs (TTRL paper: 60, PoC: 20)
    LR: float = 5e-6                    # Higher LR for LoRA GRPO (5e-7 too conservative)
    MAX_GRAD_NORM: float = 1.0

    # Evaluation episode counts (scaled for ~2-3h PoC runtime)
    BASELINE_EPISODES: int = 1           # Deterministic (temp=0), identical every run
    ACE_ONLY_EPISODES: int = 5            # Enough to show playbook evolution trajectory
    TRAINED_EVAL_EPISODES: int = 3       # Post-training eval (TTRL-only, ACE+TTRL)

    # Frozen-model inference: use vLLM offline engine for batched generation
    USE_VLLM_FOR_FROZEN: bool = True
    VLLM_GPU_UTIL_FROZEN: float = 0.95   # Push to 95% for offline batch inference
    VLLM_MAX_MODEL_LEN: int = 4096

    # Paths
    RESULTS_DIR: str = "results"
    CHECKPOINTS_DIR: str = "checkpoints"


CFG = Config()

# Adaptive config: scale down for smaller GPUs
if gpu_mem_gb < 20:
    print("Detected < 20GB GPU — reducing config for T4/L4 compatibility")
    CFG.NUM_GENERATIONS = 8
    CFG.GRPO_EPOCHS = 10
    CFG.LORA_RANK = 32
    CFG.LORA_ALPHA = 32
    CFG.VLLM_GPU_UTIL_FROZEN = 0.85

os.makedirs(CFG.RESULTS_DIR, exist_ok=True)
os.makedirs(CFG.CHECKPOINTS_DIR, exist_ok=True)

print("\nConfig:")
for k, v in vars(CFG).items():
    print(f"  {k}: {v}")

In [ ]:
# ---------------------------------------------------------------------------
# Answer parsing (reused from experiments/dc_vs_verification/run.py)
# ---------------------------------------------------------------------------

ANSWER_REGEX = re.compile(r"-?\d+(?:,\d{3})*(?:\.\d+)?")


def extract_numeric_answer(text: str) -> str:
    matches = ANSWER_REGEX.findall(text.replace(",", ""))
    if not matches:
        return text.strip()
    result = matches[-1].lstrip("0")
    return result if result else "0"


def last_boxed_only_string(string: str) -> str:
    idx = string.rfind("\\boxed")
    if idx < 0:
        idx = string.rfind("\\fbox")
    if idx < 0:
        return ""
    brace_idx = string.find("{", idx)
    if brace_idx < 0:
        return ""
    level = 0
    for i in range(brace_idx, len(string)):
        if string[i] == "{":
            level += 1
        elif string[i] == "}":
            level -= 1
            if level == 0:
                return string[idx : i + 1]
    return ""


def clean_answer(s):
    s = s.replace("\\dfrac", "\\frac")
    s = s.replace("x \\in", "")
    s = re.sub(r"\\mathbf\s*{([^}]*)}", r"\1", s)
    s = re.sub(r"\\textbf\s*{([^}]*)}", r"\1", s)
    return s


def remove_boxed(s):
    if "\\boxed " in s:
        left = "\\boxed "
        assert s[: len(left)] == left
        return s[len(left) :]
    left = "\\boxed{"
    if not s.startswith(left):
        return None
    assert s[-1] == "}"
    return clean_answer(s[len(left) : -1])


def fix_fracs(string):
    substrs = string.split("\\frac")
    new_str = substrs[0]
    if len(substrs) > 1:
        for substr in substrs[1:]:
            new_str += "\\frac"
            if substr[0] == "{":
                new_str += substr
            else:
                try:
                    assert len(substr) >= 2
                except AssertionError:
                    return string
                a, b = substr[0], substr[1]
                if b != "{":
                    new_str += "{" + a + "}{" + b + "}" + substr[2:]
                else:
                    new_str += "{" + a + "}" + b + substr[2:]
    return new_str


def fix_a_slash_b(string):
    if len(string.split("/")) != 2:
        return string
    a, b = string.split("/")
    try:
        a, b = int(a), int(b)
        assert string == "{}/{}".format(a, b)
        return "\\frac{" + str(a) + "}{" + str(b) + "}"
    except (AssertionError, ValueError):
        return string


def fix_sqrt(string):
    if "\\sqrt" not in string:
        return string
    splits = string.split("\\sqrt")
    new_string = splits[0]
    for split in splits[1:]:
        if split[0] != "{":
            new_string += "\\sqrt{" + split[0] + "}" + split[1:]
        else:
            new_string += "\\sqrt" + split
    return new_string


def remove_right_units(string):
    if "\\text{ " in string:
        splits = string.split("\\text{ ")
        assert len(splits) == 2
        return splits[0]
    return string


def strip_string(string):
    string = string.replace("\n", "")
    string = string.replace("\\!", "")
    string = string.replace("\\\\", "\\")
    string = string.replace("tfrac", "frac")
    string = string.replace("dfrac", "frac")
    string = string.replace("\\left", "")
    string = string.replace("\\right", "")
    string = string.replace("^{\\circ}", "")
    string = string.replace("^\\circ", "")
    string = string.replace("\\$", "")
    string = remove_right_units(string)
    string = string.replace("\\%", "")
    string = string.replace("%", "")
    string = string.replace(" .", " 0.")
    string = string.replace("{.", "{0.")
    if len(string) == 0:
        return string
    if string[0] == ".":
        string = "0" + string
    if len(string.split("=")) == 2:
        if len(string.split("=")[0]) <= 2:
            string = string.split("=")[1]
    string = fix_sqrt(string)
    string = string.replace(" ", "")
    string = fix_fracs(string)
    if string == "0.5":
        string = "\\frac{1}{2}"
    if string == "5.5":
        string = "\\frac{11}{2}"
    string = fix_a_slash_b(string)
    return string


def is_equiv(str1, str2, verbose=False):
    if str1 is None and str2 is None:
        return True
    if str1 is None or str2 is None:
        return False
    try:
        ss1 = strip_string(str1)
        ss2 = strip_string(str2)
        if verbose:
            print(ss1, ss2)
        return ss1 == ss2
    except Exception:
        return str1 == str2


def parse_answer(raw: str) -> str:
    """Extract answer from model output. Try \\boxed first, then #### pattern, then last number."""
    boxed = last_boxed_only_string(raw)
    if boxed:
        inner = remove_boxed(boxed)
        if inner is not None:
            return inner.strip()
    m = re.search(r"####\s*(-?[\d,]+\.?\d*)", raw)
    if m:
        return m.group(1).replace(",", "").strip()
    return extract_numeric_answer(raw)


def check_answer(predicted: str, ground_truth: str) -> bool:
    """Check if predicted answer matches ground truth."""
    if is_equiv(predicted, ground_truth):
        return True
    # AIME answers are integers 000-999; try numeric comparison
    try:
        p = int(float(predicted.replace(",", "")))
        g = int(float(ground_truth.replace(",", "")))
        return p == g
    except (ValueError, TypeError):
        return False

In [ ]:
# ---------------------------------------------------------------------------
# Component ABCs
# ---------------------------------------------------------------------------


class Generator(ABC):
    """Generates candidate solutions for a problem."""

    @abstractmethod
    def generate(self, problem: str, n: int, playbook_context: str = "") -> List[Dict]:
        """Returns list of {answer: str, raw: str, bullets_used: list}"""
        ...


class Evaluator(ABC):
    """Evaluates/selects among candidate answers."""

    @abstractmethod
    def evaluate(
        self, candidates: List[Dict], ground_truth: Optional[str] = None
    ) -> Dict:
        """Returns {selected_answer: str, reward_scores: list, metadata: dict}"""
        ...


class Curator(ABC):
    """Evolves the playbook based on results."""

    @abstractmethod
    def curate(
        self,
        playbook: Any,
        problem: str,
        solution: str,
        is_correct: bool,
        reflection: str,
    ) -> Any:
        """Returns updated playbook."""
        ...


class Trainer(ABC):
    """Updates model weights via RL."""

    @abstractmethod
    def train_step(
        self,
        prompts: List[str],
        completions: List[List[str]],
        rewards: List[List[float]],
    ) -> Dict:
        """Returns {loss: float, metrics: dict}"""
        ...


class PlaybookManager(ABC):
    """Manages playbook state and context injection."""

    @abstractmethod
    def get_context(self) -> str:
        """Returns playbook text for system prompt injection."""
        ...

    @abstractmethod
    def snapshot(self) -> Dict:
        """Returns serializable playbook state."""
        ...


class CurriculumSelector(ABC):
    """Selects/orders problems for training."""

    @abstractmethod
    def select(self, problems: List[Dict], episode: int) -> List[Dict]:
        """Returns ordered subset of problems for this episode."""
        ...


print("Component ABCs defined: Generator, Evaluator, Curator, Trainer, PlaybookManager, CurriculumSelector")

In [ ]:
# ---------------------------------------------------------------------------
# Data Loading: AIME 2024
# ---------------------------------------------------------------------------
import threading

# The correct URL is SakanaAI/ShinkaEvolve (NOT ShengranHu/ADAS which 404s)
AIME_CSV_URL = "https://raw.githubusercontent.com/SakanaAI/ShinkaEvolve/main/examples/adas_aime/AIME_Dataset_1983_2025.csv"
AIME_CSV_PATH = "AIME_Dataset_1983_2025.csv"

# Start download in background thread so it overlaps with any prior setup.
_download_complete = threading.Event()

def download_aime_data():
    """Download AIME dataset if not present."""
    if not os.path.exists(AIME_CSV_PATH):
        import urllib.request
        print(f"Downloading AIME dataset from {AIME_CSV_URL}...")
        try:
            urllib.request.urlretrieve(AIME_CSV_URL, AIME_CSV_PATH)
            print(f"Downloaded to {AIME_CSV_PATH}")
        except Exception as e:
            print(f"Primary URL failed: {e}")
            # Fallback: try loading from local repo clone if available
            local_path = Path("ShinkaEvolve/examples/adas_aime/AIME_Dataset_1983_2025.csv")
            if local_path.exists():
                import shutil
                shutil.copy(local_path, AIME_CSV_PATH)
                print(f"Copied from local repo: {local_path}")
            else:
                raise RuntimeError(
                    f"Could not download AIME dataset. Please download manually from:\n"
                    f"  {AIME_CSV_URL}\n"
                    f"and place it at: {AIME_CSV_PATH}"
                )
    _download_complete.set()

# Fire-and-forget background download
_dl_thread = threading.Thread(target=download_aime_data, daemon=True)
_dl_thread.start()


def load_aime_2024() -> List[Dict]:
    """Load AIME 2024 problems from CSV. Waits for download if needed."""
    _download_complete.wait()  # Block only if download still in progress
    problems = []
    with open(AIME_CSV_PATH, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if str(row["Year"]).strip() == "2024":
                problems.append(
                    {
                        "id": row["ID"],
                        "problem": row["problem"],
                        "answer": str(int(row["answer"])),
                    }
                )
    return problems


# Load and verify
problems = load_aime_2024()
print(f"Loaded {len(problems)} AIME 2024 problems")
assert len(problems) == 30, f"Expected 30 problems, got {len(problems)}"

# Quick answer parsing tests
assert parse_answer("The answer is \\boxed{42}") == "42"
assert parse_answer("#### 7") == "7"
assert parse_answer("The answer is 100.") == "100"
assert check_answer("42", "42") == True
assert check_answer("042", "42") == True
print("Answer parsing tests passed!")
print(f"Sample problem: {problems[0]['id']} (answer: {problems[0]['answer']})")

In [ ]:
# ---------------------------------------------------------------------------
# Condition Configs
# ---------------------------------------------------------------------------
# Each maps component names to implementations.
# These will be filled in as we implement each component.

CONDITIONS = {
    "baseline": {
        "name": "Baseline (CoT Pass@1)",
        "playbook": "null",  # NullPlaybook
        "trainer": "none",  # No training
        "evaluator": "ground_truth",  # Direct ground-truth check
        "n_generations": 1,
        "temperature": 0.0,
    },
    "ace_only": {
        "name": "ACE-only",
        "playbook": "active",  # ActivePlaybook with reflect+curate
        "trainer": "none",  # No training
        "evaluator": "majority_vote",
        "n_generations": 16,
        "temperature": 0.7,
    },
    "ttrl_only": {
        "name": "TTRL-only",
        "playbook": "null",  # NullPlaybook
        "trainer": "grpo",  # GRPO training
        "evaluator": "majority_vote",
        "n_generations": 16,
        "temperature": 0.7,
    },
    "ace_ttrl": {
        "name": "ACE+TTRL",
        "playbook": "active",  # ActivePlaybook
        "trainer": "grpo",  # GRPO training
        "evaluator": "majority_vote",
        "n_generations": 16,
        "temperature": 0.7,
    },
}

print("Condition configs defined:")
for k, v in CONDITIONS.items():
    print(
        f"  {v['name']}: playbook={v['playbook']}, trainer={v['trainer']}, eval={v['evaluator']}"
    )

In [ ]:
# ---------------------------------------------------------------------------
# Playbook, Reflect/Curate Pipeline, Evaluators, Curriculum
# ---------------------------------------------------------------------------


@dataclass
class Bullet:
    id: str
    section: str
    content: str
    helpful: int = 0
    harmful: int = 0

    def to_str(self) -> str:
        return f"[{self.id}] helpful={self.helpful} harmful={self.harmful} :: {self.content}"


class Playbook:
    """Internal playbook state with bullet management."""

    def __init__(self):
        self.bullets: List[Bullet] = []
        self._next_id: int = 1

    def add(self, section: str, content: str) -> str:
        prefix = {"STRATEGIES": "str", "COMMON_MISTAKES": "err", "SOLUTION_PATTERNS": "sol"}.get(section, "gen")
        bid = f"{prefix}-{self._next_id:05d}"
        self._next_id += 1
        self.bullets.append(Bullet(id=bid, section=section, content=content))
        return bid

    def remove(self, bid: str):
        self.bullets = [b for b in self.bullets if b.id != bid]

    def update(self, bid: str, content: str):
        for b in self.bullets:
            if b.id == bid:
                b.content = content
                return

    def tag(self, bid: str, label: str):
        for b in self.bullets:
            if b.id == bid:
                if label == "helpful":
                    b.helpful += 1
                elif label == "harmful":
                    b.harmful += 1

    def to_str(self) -> str:
        sections = defaultdict(list)
        for b in self.bullets:
            sections[b.section].append(b.to_str())
        parts = []
        for sec in ["STRATEGIES", "COMMON_MISTAKES", "SOLUTION_PATTERNS"]:
            if sections[sec]:
                parts.append(f"## {sec}")
                parts.extend(sections[sec])
        return "\n".join(parts) if parts else "(empty playbook)"

    def copy(self) -> "Playbook":
        return copy.deepcopy(self)

    @property
    def size(self) -> int:
        return len(self.bullets)

    def snapshot(self) -> Dict:
        return {
            "bullets": [{"id": b.id, "section": b.section, "content": b.content,
                          "helpful": b.helpful, "harmful": b.harmful} for b in self.bullets],
            "next_id": self._next_id,
        }

    @classmethod
    def from_snapshot(cls, data: Dict) -> "Playbook":
        pb = cls()
        pb._next_id = data.get("next_id", 1)
        for bd in data.get("bullets", []):
            pb.bullets.append(Bullet(**bd))
        return pb


def make_initial_playbook() -> Playbook:
    pb = Playbook()
    pb.add("STRATEGIES", "AIME problems have integer answers from 000 to 999. Always give a non-negative integer.")
    pb.add("STRATEGIES", "Break complex problems into smaller sub-problems and solve each step carefully.")
    pb.add("COMMON_MISTAKES", "Watch for off-by-one errors in counting and combinatorics problems.")
    return pb


# ---------------------------------------------------------------------------
# PlaybookManager implementations
# ---------------------------------------------------------------------------

REFLECT_SYSTEM = (
    "You are a math reasoning analyst. Analyze the solution and whether playbook strategies helped.\n"
    'For each bullet ID used, output a JSON line: {"id": "str-00001", "tag": "helpful"}\n'
    "Tags: helpful, harmful, neutral.\n"
    "End with a brief reflection about what mathematical insight was key."
)

def _build_curate_system(max_bullets, current_size):
    return (
        "You are a playbook curator for math competition solving. Based on the reflection, "
        "propose operations to improve the playbook.\n"
        "Output a JSON array of operations:\n"
        '[{"op": "ADD", "section": "STRATEGIES", "content": "new insight"},\n'
        ' {"op": "UPDATE", "id": "str-00001", "content": "refined text"},\n'
        ' {"op": "DELETE", "id": "err-00002"}]\n'
        f"Sections: STRATEGIES, COMMON_MISTAKES, SOLUTION_PATTERNS\n"
        f"Max bullets: {max_bullets}. Current: {current_size}.\n"
        "Only propose operations clearly supported by the reflection. Keep it minimal."
    )


class NullPlaybook(PlaybookManager):
    def get_context(self) -> str:
        return ""
    def snapshot(self) -> Dict:
        return {"type": "null"}
    def reflect_and_curate(self, problem, solution, is_correct, candidates, generate_fn=None):
        pass
    def batch_reflect_and_curate(self, items, batch_generate_fn=None):
        pass


class ActivePlaybook(PlaybookManager):
    """Evolving playbook with reflect+curate pipeline."""

    def __init__(self, generate_fn):
        self.playbook = make_initial_playbook()
        self._generate_fn = generate_fn

    def get_context(self) -> str:
        if self.playbook.size == 0:
            return ""
        return f"\nPLAYBOOK (use these strategies, reference IDs like [str-00001]):\n{self.playbook.to_str()}"

    def snapshot(self) -> Dict:
        return {"type": "active", "playbook": self.playbook.snapshot()}

    def reflect_and_curate(self, problem, solution, is_correct, candidates, generate_fn=None):
        """Single-problem reflect+curate (fallback, prefer batch_reflect_and_curate)."""
        fn = generate_fn or self._generate_fn
        best = candidates[0] if candidates else {"raw": solution, "bullets_used": []}
        bullets_used = best.get("bullets_used", [])
        raw_response = best.get("raw", solution)

        feedback = "CORRECT" if is_correct else "INCORRECT"
        bullets_text = "\n".join(f"  {b.to_str()}" for b in self.playbook.bullets if b.id in bullets_used)
        if not bullets_text:
            bullets_text = "  (none referenced)"

        reflect_user = f"Problem: {problem}\n\nSolution:\n{raw_response[:2000]}\n\nResult: {feedback}\n\nBullets referenced:\n{bullets_text}"
        reflection = fn(REFLECT_SYSTEM, reflect_user, temperature=0.3, max_tokens=512)

        self._apply_reflect_tags(reflection, bullets_used, is_correct)

        pb_text = self.playbook.to_str()
        curate_system = _build_curate_system(CFG.MAX_BULLETS, self.playbook.size)
        curate_user = f"Question: {problem}\nCurrent playbook:\n{pb_text}\n\nReflection:\n{reflection}"
        curate_raw = fn(curate_system, curate_user, temperature=0.4, max_tokens=512)
        self._apply_curate_ops(curate_raw)

    def batch_reflect_and_curate(self, items: List[Dict], batch_generate_fn=None):
        """Batched reflect+curate for all problems in an episode.

        Instead of 60 sequential LLM calls (30 reflect + 30 curate),
        does 2 batched calls. ~15x speedup on vLLM.

        items: list of {problem, solution, is_correct, candidates}
        batch_generate_fn: callable(messages_list, temp, max_tokens) -> list[str]
        """
        if not items:
            return
        batch_fn = batch_generate_fn
        if batch_fn is None:
            return  # No batch function available

        # --- Step 1: Build all reflect prompts ---
        reflect_messages = []
        items_meta = []  # Track bullets_used per item
        for item in items:
            best = item["candidates"][0] if item["candidates"] else {"raw": item["solution"], "bullets_used": []}
            bullets_used = best.get("bullets_used", [])
            raw_response = best.get("raw", item["solution"])
            feedback = "CORRECT" if item["is_correct"] else "INCORRECT"
            bullets_text = "\n".join(f"  {b.to_str()}" for b in self.playbook.bullets if b.id in bullets_used)
            if not bullets_text:
                bullets_text = "  (none referenced)"
            reflect_user = f"Problem: {item['problem']}\n\nSolution:\n{raw_response[:2000]}\n\nResult: {feedback}\n\nBullets referenced:\n{bullets_text}"
            reflect_messages.append((REFLECT_SYSTEM, reflect_user))
            items_meta.append({"bullets_used": bullets_used, "is_correct": item["is_correct"]})

        # --- Step 2: Batch reflect (ONE vLLM call for all 30) ---
        reflections = batch_fn(reflect_messages, 0.3, 512)

        # --- Step 3: Apply tags from all reflections ---
        for reflection, meta in zip(reflections, items_meta):
            self._apply_reflect_tags(reflection, meta["bullets_used"], meta["is_correct"])

        # --- Step 4: Build all curate prompts ---
        pb_text = self.playbook.to_str()  # Same playbook state for all curates in this episode
        curate_system = _build_curate_system(CFG.MAX_BULLETS, self.playbook.size)
        curate_messages = []
        for item, reflection in zip(items, reflections):
            curate_user = f"Question: {item['problem']}\nCurrent playbook:\n{pb_text}\n\nReflection:\n{reflection}"
            curate_messages.append((curate_system, curate_user))

        # --- Step 5: Batch curate (ONE vLLM call for all 30) ---
        curate_results = batch_fn(curate_messages, 0.4, 512)

        # --- Step 6: Apply all curate ops ---
        for curate_raw in curate_results:
            self._apply_curate_ops(curate_raw)

        # Safety: if curate emptied the playbook, restore initial
        if self.playbook.size == 0:
            old_next = self.playbook._next_id
            self.playbook = make_initial_playbook()
            self.playbook._next_id = old_next

    def _apply_reflect_tags(self, reflection: str, bullets_used: List[str], is_correct: bool):
        """Parse and apply tags from a reflection."""
        tags = {}
        for m in re.finditer(r'"id"\s*:\s*"([^"]+)".*?"tag"\s*:\s*"(helpful|harmful|neutral)"', reflection):
            bid, tag = m.group(1), m.group(2)
            if bid in bullets_used:
                tags[bid] = tag
        if not tags and bullets_used:
            default_tag = "helpful" if is_correct else "harmful"
            for bid in bullets_used:
                tags[bid] = default_tag
        for bid, tag in tags.items():
            self.playbook.tag(bid, tag)

    def _apply_curate_ops(self, curate_raw: str):
        """Parse and apply curate operations."""
        json_match = re.search(r"\[.*\]", curate_raw, re.DOTALL)
        if not json_match:
            return
        try:
            ops = json.loads(json_match.group())
        except json.JSONDecodeError:
            return
        for op in ops:
            try:
                if op.get("op") == "ADD" and self.playbook.size < CFG.MAX_BULLETS:
                    self.playbook.add(op.get("section", "STRATEGIES"), op.get("content", ""))
                elif op.get("op") == "UPDATE" and op.get("id"):
                    self.playbook.update(op["id"], op.get("content", ""))
                elif op.get("op") == "DELETE" and op.get("id"):
                    self.playbook.remove(op["id"])
            except Exception:
                pass


# ---------------------------------------------------------------------------
# Majority vote helper
# ---------------------------------------------------------------------------

def majority_vote(answers: List[str]) -> Tuple[str, float]:
    counter = Counter()
    for a in answers:
        try:
            normalized = str(int(float(a.replace(",", ""))))
        except (ValueError, TypeError):
            normalized = a.strip()
        counter[normalized] += 1
    if not counter:
        return "", 0.0
    winner, count = counter.most_common(1)[0]
    return winner, count / len(answers)


# ---------------------------------------------------------------------------
# Evaluator implementations
# ---------------------------------------------------------------------------

class MajorityVoteEvaluator(Evaluator):
    def evaluate(self, candidates: List[Dict], ground_truth: Optional[str] = None) -> Dict:
        answers = [c["answer"] for c in candidates]
        winner, confidence = majority_vote(answers)
        rewards = []
        for c in candidates:
            try:
                norm = str(int(float(c["answer"].replace(",", ""))))
            except (ValueError, TypeError):
                norm = c["answer"].strip()
            rewards.append(1.0 if norm == winner else 0.0)
        return {"selected_answer": winner, "confidence": confidence,
                "reward_scores": rewards, "metadata": {"vote_distribution": dict(Counter(answers).most_common())}}


class GroundTruthEvaluator(Evaluator):
    def evaluate(self, candidates: List[Dict], ground_truth: Optional[str] = None) -> Dict:
        if not candidates:
            return {"selected_answer": "", "reward_scores": [], "metadata": {}}
        answer = candidates[0]["answer"]
        correct = check_answer(answer, ground_truth) if ground_truth else False
        return {"selected_answer": answer, "reward_scores": [1.0 if correct else 0.0], "metadata": {"correct": correct}}


class IdentityCurriculum(CurriculumSelector):
    def select(self, problems: List[Dict], episode: int) -> List[Dict]:
        return problems


# Smoke tests
pb = make_initial_playbook()
assert pb.size == 3
null_pb = NullPlaybook()
assert null_pb.get_context() == ""
snap = pb.snapshot()
pb2 = Playbook.from_snapshot(snap)
assert pb2.size == pb.size
print(f"Playbook ({pb.size} bullets), NullPlaybook, snapshot round-trip: OK")

In [ ]:
# ---------------------------------------------------------------------------
# Model Loading + GRPO Training Setup
# ---------------------------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA configuration
# RL training needs higher rank than SFT (veRL docs recommend >= 32 for 7B).
# MLP modules (gate_proj, up_proj, down_proj) are critical for math reasoning.
# rsLoRA uses α/√r scaling to prevent gradient collapse at high ranks.
peft_config = LoraConfig(
    r=CFG.LORA_RANK,
    lora_alpha=CFG.LORA_ALPHA,
    target_modules=CFG.LORA_MODULES,
    task_type="CAUSAL_LM",
    bias="none",
    use_rslora=True,
)

# System prompt builder
def build_system_prompt(playbook_context: str = "") -> str:
    base = (
        "You are an expert math competition solver. Solve the problem step-by-step.\n"
        "Show all your work clearly. At the end, put your final integer answer inside \\boxed{}.\n"
        "AIME answers are always integers from 0 to 999.\n"
    )
    if playbook_context:
        base += playbook_context
    return base

# Format prompts for the model using chat template
def format_prompt(problem: str, playbook_context: str = "") -> str:
    """Format a single problem as a chat-template prompt string."""
    system = build_system_prompt(playbook_context)
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Solve this AIME problem:\n\n{problem}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# ---------------------------------------------------------------------------
# vLLM Offline Engine for frozen-model conditions
# ---------------------------------------------------------------------------

_vllm_engine = {"llm": None}

def init_vllm_engine():
    """Initialize vLLM offline engine for frozen-model inference."""
    from vllm import LLM
    print(f"Initializing vLLM offline engine (gpu_util={CFG.VLLM_GPU_UTIL_FROZEN})...")
    t0 = time.time()
    _vllm_engine["llm"] = LLM(
        model=CFG.MODEL_NAME,
        dtype="bfloat16",
        gpu_memory_utilization=CFG.VLLM_GPU_UTIL_FROZEN,
        max_model_len=CFG.VLLM_MAX_MODEL_LEN,
        max_num_seqs=512,              # Handles 30×16=480 seq batch
        enable_prefix_caching=True,    # Shared system prompt → ~10x reuse
        enable_chunked_prefill=True,   # +15-25% throughput for mixed-length
        enforce_eager=True,            # Avoid CUDA graph memory leaks
        seed=42,
    )
    print(f"vLLM engine ready in {time.time()-t0:.1f}s")

def shutdown_vllm_engine():
    """Shut down vLLM engine and free GPU memory for GRPO training."""
    if _vllm_engine["llm"] is not None:
        from vllm.distributed.parallel_state import destroy_model_parallel
        del _vllm_engine["llm"]
        _vllm_engine["llm"] = None
        destroy_model_parallel()
        # Synchronize first, then double gc.collect for cyclic refs
        torch.cuda.synchronize()
        gc.collect()
        gc.collect()
        torch.cuda.empty_cache()
        print("vLLM engine shut down, GPU memory freed.")


def vllm_batch_generate(prompts: List[str], n: int, temperature: float,
                         max_tokens: int = None) -> List[List[Dict]]:
    """Generate n completions per prompt using vLLM offline engine."""
    if max_tokens is None:
        max_tokens = CFG.MAX_COMPLETION_LENGTH
    from vllm import SamplingParams
    llm = _vllm_engine["llm"]
    assert llm is not None, "vLLM engine not initialized."

    sampling_params = SamplingParams(
        n=n,
        temperature=max(temperature, 0.01) if n > 1 else 0.01,
        max_tokens=max_tokens,
        stop=["<|endoftext|>", "<|im_end|>"],
    )
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)

    all_candidates = []
    for request_output in outputs:
        candidates = []
        for completion in request_output.outputs:
            text = completion.text
            answer = parse_answer(text)
            bullets_used = re.findall(r"\[(str|err|sol|gen)-\d{5}\]", text)
            candidates.append({
                "answer": answer, "raw": text,
                "bullets_used": list(set(bullets_used)),
            })
        all_candidates.append(candidates)
    return all_candidates


def vllm_batch_text_generate(messages_list: List[Tuple[str, str]],
                              temperature: float = 0.3,
                              max_tokens: int = 512) -> List[str]:
    """Batch text generation for multiple (system, user) pairs.

    Used for batched reflect/curate: sends all 30 prompts in ONE vLLM call
    instead of 30 sequential calls. ~15x speedup for reflect/curate phase.
    """
    from vllm import SamplingParams
    llm = _vllm_engine["llm"]
    if llm is None:
        return ['{"id": "none", "tag": "neutral"}'] * len(messages_list)

    prompts = []
    for system, user in messages_list:
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

    sampling_params = SamplingParams(
        n=1, temperature=max(temperature, 0.01), max_tokens=max_tokens,
        stop=["<|endoftext|>", "<|im_end|>"],
    )
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
    return [o.outputs[0].text.strip() for o in outputs]


def vllm_single_generate(system: str, user: str, temperature: float = 0.3,
                          max_tokens: int = 1024) -> str:
    """Single-completion generation via vLLM (fallback, prefer batch)."""
    results = vllm_batch_text_generate([(system, user)], temperature, max_tokens)
    return results[0]


# ---------------------------------------------------------------------------
# Synchronous generation for playbook reflect/curate (HF fallback)
# ---------------------------------------------------------------------------
_sync_model_ref = {"model": None}

def sync_generate(system: str, user: str, temperature: float = 0.3, max_tokens: int = 1024) -> str:
    """Synchronous LLM call for reflect/curate. Prefers vLLM, falls back to HF."""
    if _vllm_engine["llm"] is not None:
        return vllm_single_generate(system, user, temperature, max_tokens)

    model = _sync_model_ref["model"]
    if model is None:
        return '{"id": "none", "tag": "neutral"}\nNo model loaded for reflection.'
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)
    inputs = inputs.to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            inputs, max_new_tokens=max_tokens,
            temperature=max(temperature, 0.01),
            do_sample=temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response.strip()

# ---------------------------------------------------------------------------
# GRPO reward function: majority-voting based (TTRL-style)
# ---------------------------------------------------------------------------
def ttrl_reward_fn(completions, **kwargs) -> list[float]:
    """Compute majority-vote reward for a batch of completions."""
    answers = [parse_answer(c) for c in completions]
    winner, confidence = majority_vote(answers)
    rewards = []
    for a in answers:
        try:
            norm = str(int(float(a.replace(",", ""))))
        except (ValueError, TypeError):
            norm = a.strip()
        rewards.append(1.0 if norm == winner else 0.0)
    return rewards

# ---------------------------------------------------------------------------
# ACE+TTRL reward: majority-vote + deferred playbook reflect/curate
# ---------------------------------------------------------------------------
_ace_ttrl_state = {
    "playbook_mgr": None,
    "problem_lookup": {},
    "episode_stats": [],
    "pending_curate": [],
}

def ace_ttrl_reward_fn(prompts, completions, **kwargs) -> list[float]:
    """Majority-vote reward with deferred playbook evolution."""
    answers = [parse_answer(c) for c in completions]
    winner, confidence = majority_vote(answers)

    rewards = []
    for a in answers:
        try:
            norm = str(int(float(a.replace(",", ""))))
        except (ValueError, TypeError):
            norm = a.strip()
        rewards.append(confidence if norm == winner else 0.0)

    pb_mgr = _ace_ttrl_state["playbook_mgr"]
    if pb_mgr is not None and prompts:
        prompt_key = prompts[0] if isinstance(prompts, list) else str(prompts)
        problem_dict = _ace_ttrl_state["problem_lookup"].get(prompt_key, None)
        ground_truth = problem_dict["answer"] if problem_dict else None
        is_correct = check_answer(winner, ground_truth) if ground_truth else False

        candidates = []
        for c_text, a in zip(completions, answers):
            bullets_used = re.findall(r"\[(str|err|sol|gen)-\d{5}\]", c_text)
            candidates.append({"answer": a, "raw": c_text, "bullets_used": list(set(bullets_used))})

        # Tag bullets based on correctness (no LLM needed)
        for c in candidates:
            for bid in c.get("bullets_used", []):
                pb_mgr.playbook.tag(bid, "helpful" if is_correct else "harmful")

        _ace_ttrl_state["pending_curate"].append({
            "problem": problem_dict["problem"] if problem_dict else "(unknown)",
            "solution": winner,
            "is_correct": is_correct,
            "candidates": candidates,
        })
        _ace_ttrl_state["episode_stats"].append({
            "confidence": confidence, "is_correct": is_correct,
            "pb_size": pb_mgr.playbook.size if hasattr(pb_mgr, "playbook") else 0,
        })

    return rewards

# ---------------------------------------------------------------------------
# GRPO Training configs
# ---------------------------------------------------------------------------

# GRPO config for TTRL-only (Phase 2): 20 epochs in a single .train() call
grpo_config = GRPOConfig(
    output_dir=CFG.CHECKPOINTS_DIR,
    num_train_epochs=CFG.GRPO_EPOCHS,
    per_device_train_batch_size=2,       # Increased from 1 (A100 has headroom)
    gradient_accumulation_steps=2,       # Decreased from 4 (same effective batch=4)
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=CFG.LR,
    max_grad_norm=CFG.MAX_GRAD_NORM,
    bf16=True,
    logging_steps=1,
    save_strategy="no",
    num_generations=CFG.NUM_GENERATIONS,
    generation_batch_size=CFG.NUM_GENERATIONS * 2,  # Process 2 prompts in parallel
    max_completion_length=CFG.MAX_COMPLETION_LENGTH,
    temperature=0.6,
    top_p=0.95,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.4,     # Increased from 0.3 for better gen throughput
    vllm_enable_sleep_mode=True,
    beta=CFG.KL_COEFF,
    report_to="none",
)

# GRPO config for ACE+TTRL (Phase 3): 1 epoch per manual loop iteration
ace_ttrl_grpo_config = GRPOConfig(
    output_dir=CFG.CHECKPOINTS_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=CFG.LR,
    max_grad_norm=CFG.MAX_GRAD_NORM,
    bf16=True,
    logging_steps=1,
    save_strategy="no",
    num_generations=CFG.NUM_GENERATIONS,
    generation_batch_size=CFG.NUM_GENERATIONS * 2,
    max_completion_length=CFG.MAX_COMPLETION_LENGTH,
    temperature=0.6,
    top_p=0.95,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.4,
    vllm_enable_sleep_mode=True,
    beta=CFG.KL_COEFF,
    report_to="none",
)

class NullTrainer(Trainer):
    def train_step(self, prompts, completions, rewards) -> Dict:
        return {"loss": 0.0, "metrics": {}}

print("GRPO configs ready:")
print(f"  TTRL-only: {grpo_config.num_train_epochs} epochs, temp={grpo_config.temperature}")
print(f"  ACE+TTRL:   {ace_ttrl_grpo_config.num_train_epochs} epoch/iter x {CFG.GRPO_EPOCHS} manual epochs")
print(f"  per_device_batch={grpo_config.per_device_train_batch_size}, grad_accum={grpo_config.gradient_accumulation_steps}")
print(f"  num_generations: {grpo_config.num_generations}, generation_batch_size: {grpo_config.generation_batch_size}")
print(f"  max_completion_length: {grpo_config.max_completion_length}, vllm_gpu_util: {grpo_config.vllm_gpu_memory_utilization}")
print(f"  LoRA rank: {peft_config.r}, alpha: {peft_config.lora_alpha}, rsLoRA: {peft_config.use_rslora}")
print(f"  LoRA modules: {peft_config.target_modules}")
print(f"  LR: {grpo_config.learning_rate}")
print(f"Tokenizer: pad_token={tokenizer.pad_token!r}")

In [ ]:
# ---------------------------------------------------------------------------
# Co-Evolution Experiment Loop
# ---------------------------------------------------------------------------
from datasets import Dataset
from transformers import TrainerState


def run_frozen_condition(condition_key: str, problems: List[Dict],
                         num_episodes: int) -> Dict:
    """Run a non-training condition (baseline or ACE-only) using vLLM offline engine.

    Uses vLLM batch generation for high GPU utilization: all problems' candidates
    are generated in a single batched call per episode via continuous batching
    and PagedAttention. This is 5-10x faster than sequential HF generate.

    Reflect/curate uses batch_reflect_and_curate: 2 batched vLLM calls per
    episode instead of 60 sequential calls (~15x speedup).
    """
    cond = CONDITIONS[condition_key]
    print(f"\n{'='*60}")
    print(f"Running: {cond['name']} ({num_episodes} episodes)")
    print(f"{'='*60}")

    use_playbook = cond["playbook"] == "active"
    n_gen = cond["n_generations"]
    temp = cond["temperature"]

    # Initialize playbook (uses vLLM engine for reflect/curate LLM calls)
    if use_playbook:
        playbook_mgr = ActivePlaybook(vllm_single_generate)
    else:
        playbook_mgr = NullPlaybook()

    # Initialize evaluator
    if cond["evaluator"] == "majority_vote":
        evaluator = MajorityVoteEvaluator()
    else:
        evaluator = GroundTruthEvaluator()

    all_episode_results = []
    playbook_snapshots = []

    for episode in range(num_episodes):
        episode_start = time.time()
        episode_correct = 0
        episode_total = 0
        episode_details = []

        # Build all prompts with current playbook context
        playbook_context = playbook_mgr.get_context()
        prompts = [format_prompt(p["problem"], playbook_context) for p in problems]

        # Batch generate: ONE vLLM call for all problems x N candidates.
        # vLLM handles continuous batching internally, fully utilizing the GPU.
        all_candidates = vllm_batch_generate(prompts, n=n_gen, temperature=temp)

        # Evaluate all problems and collect items for batched reflect/curate
        curate_items = []
        for p_idx, (problem, candidates) in enumerate(zip(problems, all_candidates)):
            eval_result = evaluator.evaluate(candidates, ground_truth=problem["answer"])
            selected = eval_result["selected_answer"]
            is_correct = check_answer(selected, problem["answer"])

            if use_playbook:
                curate_items.append({
                    "problem": problem["problem"],
                    "solution": selected,
                    "is_correct": is_correct,
                    "candidates": candidates,
                })

            episode_correct += int(is_correct)
            episode_total += 1
            episode_details.append({
                "problem_id": problem["id"],
                "selected_answer": selected,
                "ground_truth": problem["answer"],
                "correct": is_correct,
                "n_candidates": len(candidates),
                "confidence": eval_result.get("confidence", None),
            })

        # Batched reflect+curate: 2 vLLM calls for all 30 problems
        # instead of 60 sequential calls (~15x speedup)
        if use_playbook and curate_items:
            playbook_mgr.batch_reflect_and_curate(curate_items, vllm_batch_text_generate)

        playbook_snapshots.append(playbook_mgr.snapshot())
        episode_acc = episode_correct / episode_total if episode_total > 0 else 0.0
        episode_time = time.time() - episode_start

        all_episode_results.append({
            "episode": episode,
            "accuracy": episode_acc,
            "correct": episode_correct,
            "total": episode_total,
            "time_s": episode_time,
            "details": episode_details,
        })

        pb_size = playbook_mgr.playbook.size if hasattr(playbook_mgr, "playbook") else 0
        print(f"  Episode {episode+1}/{num_episodes}: "
              f"acc={episode_acc:.1%} ({episode_correct}/{episode_total}) "
              f"pb_size={pb_size} time={episode_time:.0f}s")

    return {
        "condition": condition_key,
        "name": cond["name"],
        "episodes": all_episode_results,
        "playbook_snapshots": playbook_snapshots,
        "training_metrics": [],
    }


def evaluate_trained_model(condition_key: str, problems: List[Dict],
                           grpo_trainer, tokenizer,
                           num_episodes: int) -> Dict:
    """Evaluate post-GRPO model by merging LoRA + re-initializing vLLM.

    Instead of slow HF model.generate() (30 problems × 4 batches × 3 episodes
    = 360 sequential calls, ~35 min/episode), we:
    1. Save LoRA adapter
    2. Merge into base model on CPU
    3. Re-init vLLM offline engine with merged weights
    4. Use vllm_batch_generate for 10-15x speedup (~3-5 min/episode)
    """
    from peft import PeftModel
    from vllm import LLM, SamplingParams

    cond = CONDITIONS[condition_key]
    print(f"\n{'='*60}")
    print(f"Evaluating: {cond['name']} (post-training, {num_episodes} episodes)")
    print(f"{'='*60}")

    n_gen = cond["n_generations"]
    temp = cond["temperature"]
    evaluator = MajorityVoteEvaluator()

    # Step 1: Save LoRA adapter
    adapter_path = os.path.join(CFG.CHECKPOINTS_DIR, f"{condition_key}_adapter")
    grpo_trainer.save_model(adapter_path)
    print(f"  Adapter saved to {adapter_path}")

    # Step 2: Merge LoRA into base model (on CPU to avoid GPU OOM)
    print("  Merging LoRA adapter with base model (CPU)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        CFG.MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cpu",
    )
    peft_model = PeftModel.from_pretrained(base_model, adapter_path)
    merged_model = peft_model.merge_and_unload()
    merged_path = os.path.join(CFG.CHECKPOINTS_DIR, f"{condition_key}_merged")
    merged_model.save_pretrained(merged_path)
    tokenizer.save_pretrained(merged_path)
    del base_model, peft_model, merged_model
    print(f"  Merged model saved to {merged_path}")

    # Step 3: Free GPU memory from trainer
    # Note: we don't delete the trainer itself — caller may still need it
    torch.cuda.synchronize()
    gc.collect()
    gc.collect()
    torch.cuda.empty_cache()

    # Step 4: Re-init vLLM with merged model
    print("  Initializing vLLM offline engine with merged model...")
    eval_llm = LLM(
        model=merged_path,
        dtype="bfloat16",
        gpu_memory_utilization=CFG.VLLM_GPU_UTIL_FROZEN,
        max_model_len=CFG.VLLM_MAX_MODEL_LEN,
        max_num_seqs=512,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
        enforce_eager=True,
        seed=42,
    )

    # Step 5: Run evaluation episodes with vLLM batch generation
    sampling_params = SamplingParams(
        n=n_gen,
        temperature=max(temp, 0.01),
        max_tokens=CFG.MAX_COMPLETION_LENGTH,
        stop=["<|endoftext|>", "<|im_end|>"],
    )

    all_episode_results = []

    for episode in range(num_episodes):
        episode_start = time.time()
        episode_correct = 0
        episode_total = 0
        episode_details = []

        prompts = [format_prompt(p["problem"]) for p in problems]
        outputs = eval_llm.generate(prompts, sampling_params, use_tqdm=False)

        for problem, output in zip(problems, outputs):
            candidates = []
            for completion in output.outputs:
                text = completion.text
                candidates.append({
                    "answer": parse_answer(text),
                    "raw": text,
                    "bullets_used": [],
                })

            eval_result = evaluator.evaluate(candidates, ground_truth=problem["answer"])
            selected = eval_result["selected_answer"]
            is_correct = check_answer(selected, problem["answer"])

            episode_correct += int(is_correct)
            episode_total += 1
            episode_details.append({
                "problem_id": problem["id"],
                "selected_answer": selected,
                "ground_truth": problem["answer"],
                "correct": is_correct,
                "n_candidates": len(candidates),
                "confidence": eval_result.get("confidence", None),
            })

        episode_acc = episode_correct / episode_total if episode_total > 0 else 0.0
        episode_time = time.time() - episode_start

        all_episode_results.append({
            "episode": episode,
            "accuracy": episode_acc,
            "correct": episode_correct,
            "total": episode_total,
            "time_s": episode_time,
            "details": episode_details,
        })

        print(f"  Episode {episode+1}/{num_episodes}: "
              f"acc={episode_acc:.1%} ({episode_correct}/{episode_total}) "
              f"time={episode_time:.0f}s")

    # Step 6: Cleanup eval vLLM engine
    from vllm.distributed.parallel_state import destroy_model_parallel
    del eval_llm
    destroy_model_parallel()
    torch.cuda.synchronize()
    gc.collect()
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "condition": condition_key,
        "name": cond["name"],
        "episodes": all_episode_results,
        "playbook_snapshots": [{"type": "null"}] * num_episodes,
        "training_metrics": [],
    }


def reset_trainer_state(trainer):
    """Reset trainer internal state for safe repeated .train() calls.

    Calling .train() multiple times on a HuggingFace Trainer can fail because
    the LR scheduler is exhausted and the step counter doesn't reset.
    This resets state while preserving the optimizer (Adam momentum) and model
    weights, so training continues smoothly with updated data.
    """
    trainer.state = TrainerState()
    trainer.lr_scheduler = None     # Force recreation in next train() call
    # Keep trainer.optimizer intact — Adam momentum carries over for stable training
    # Keep trainer.model intact — weights carry over between epochs


def rule_based_curate(playbook_mgr, pending_items):
    """Reward-aware playbook evolution for ACE+TTRL (no LLM calls needed).

    The reward fn already tags bullets as helpful/harmful based on correctness.
    This function:
    1. Prunes toxic bullets (harmful >> helpful)
    2. Amplifies bullets from high-reward completions (reward→memory coupling)
    3. Enforces MAX_BULLETS cap by pruning lowest-signal bullets
    """
    pb = playbook_mgr.playbook

    # 1. Prune bullets where harmful significantly exceeds helpful
    to_remove = [b.id for b in pb.bullets if b.harmful > b.helpful + 2]
    for bid in to_remove:
        pb.remove(bid)

    # 2. Amplify high-reward bullets: correct solutions' bullets get bonus tags
    for item in pending_items:
        if item["is_correct"]:
            # Find candidate with most bullet references (best use of playbook)
            best = max(item["candidates"], key=lambda c: len(c.get("bullets_used", [])))
            for bid in best.get("bullets_used", []):
                pb.tag(bid, "helpful")  # Bonus tag: reward→memory coupling

    # 3. Enforce MAX_BULLETS cap via pruning lowest-signal bullets
    while pb.size > CFG.MAX_BULLETS:
        worst = min(pb.bullets, key=lambda b: b.helpful - b.harmful)
        pb.remove(worst.id)

    # Safety: if curate emptied the playbook, restore initial
    if pb.size == 0:
        old_next = pb._next_id
        playbook_mgr.playbook = make_initial_playbook()
        playbook_mgr.playbook._next_id = old_next


# ---------------------------------------------------------------------------
# Run all 4 conditions
# ---------------------------------------------------------------------------
print("=" * 60)
print("TTRL + ACE Co-Evolution Experiment (PoC Scale)")
print(f"Model: {CFG.MODEL_NAME}")
print(f"GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)")
print(f"Problems: {len(problems)} AIME 2024")
print(f"GRPO epochs: {CFG.GRPO_EPOCHS}")
print(f"Generations/prompt: {CFG.NUM_GENERATIONS}")
print(f"LoRA: rank={CFG.LORA_RANK}, modules={len(CFG.LORA_MODULES)}, rsLoRA=True")
print(f"Eval: baseline={CFG.BASELINE_EPISODES}ep, ace_only={CFG.ACE_ONLY_EPISODES}ep, "
      f"trained={CFG.TRAINED_EVAL_EPISODES}ep")
print(f"TF32: {torch.backends.cuda.matmul.allow_tf32}")
print("=" * 60)

all_results = {}

# ===================================================================
# Phase 1: Frozen-model conditions (Baseline + ACE-only)
# Uses vLLM offline engine at 95% GPU utilization for maximum throughput.
# All 30 problems x N candidates generated in a single batched call.
# Prefix caching reuses system prompt KV across all problems.
# Chunked prefill interleaves prefill with decode for +15-25% throughput.
# ===================================================================
print("\n[Phase 1] Initializing vLLM offline engine for frozen conditions...")
init_vllm_engine()

# --- Condition 1: Baseline (frozen, 1 gen, ground-truth eval) ---
# Deterministic (temp=0), so 1 episode is sufficient
result_baseline = run_frozen_condition("baseline", problems,
                                        num_episodes=CFG.BASELINE_EPISODES)
all_results["baseline"] = result_baseline

with open(os.path.join(CFG.RESULTS_DIR, "baseline.json"), "w") as f:
    json.dump(result_baseline, f, indent=2, default=str)
print(f"Baseline saved. Final acc: {result_baseline['episodes'][-1]['accuracy']:.1%}")

# --- Condition 2: ACE-only (frozen, playbook evolution, maj vote) ---
result_ace = run_frozen_condition("ace_only", problems,
                                  num_episodes=CFG.ACE_ONLY_EPISODES)
all_results["ace_only"] = result_ace

with open(os.path.join(CFG.RESULTS_DIR, "ace_only.json"), "w") as f:
    json.dump(result_ace, f, indent=2, default=str)
print(f"ACE-only saved. Final acc: {result_ace['episodes'][-1]['accuracy']:.1%}")

# Shut down vLLM engine to free GPU for GRPO training
shutdown_vllm_engine()

# ===================================================================
# Phase 2: TTRL-only (GRPO training with majority-vote reward)
# vLLM runs in colocate mode at 40% GPU, training uses the other 60%.
# GRPOTrainer handles vLLM sleep/wake automatically: wake for generation,
# sleep for training, repeat for 20 epochs.
# batch_size=2 with grad_accum=2 for better GPU utilization.
# ===================================================================
print(f"\n[Phase 2] Initializing GRPOTrainer for TTRL-only ({CFG.GRPO_EPOCHS} epochs)...")

train_prompts = [format_prompt(p["problem"]) for p in problems]
train_dataset = Dataset.from_dict({"prompt": train_prompts})

grpo_trainer_ttrl = GRPOTrainer(
    model=CFG.MODEL_NAME,
    reward_funcs=ttrl_reward_fn,
    args=grpo_config,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)
print(f"GRPOTrainer initialized (TTRL-only, {grpo_config.num_train_epochs} epochs, "
      f"temp={grpo_config.temperature}, top_p={grpo_config.top_p})")

print("Starting TTRL-only training...")
try:
    grpo_trainer_ttrl.train()
    print("TTRL-only training complete!")
except Exception as e:
    print(f"TTRL-only training error: {e}")
    import traceback; traceback.print_exc()

# Evaluate the trained model (merge LoRA + vLLM for 10-15x speedup)
result_ttrl = evaluate_trained_model(
    "ttrl_only", problems, grpo_trainer_ttrl, tokenizer,
    num_episodes=CFG.TRAINED_EVAL_EPISODES,
)
all_results["ttrl_only"] = result_ttrl

with open(os.path.join(CFG.RESULTS_DIR, "ttrl_only.json"), "w") as f:
    json.dump(result_ttrl, f, indent=2, default=str)
print(f"TTRL-only saved. Final acc: {result_ttrl['episodes'][-1]['accuracy']:.1%}")

# Save TTRL checkpoint
grpo_trainer_ttrl.save_model(os.path.join(CFG.CHECKPOINTS_DIR, "ttrl_only_final"))

# Clean up
del grpo_trainer_ttrl
torch.cuda.empty_cache()

# ===================================================================
# Phase 3: ACE+TTRL (co-evolution of weights + playbook)
#
# KEY DESIGN: Manual epoch loop with playbook injection.
# Each epoch:
#   1. Rebuild training prompts with CURRENT playbook context
#   2. Reset trainer state (scheduler/step counter) for clean epoch
#   3. Run 1 GRPO epoch (vLLM generates with playbook in prompts)
#   4. Reward fn tags bullets as helpful/harmful based on correctness
#   5. After epoch: rule-based curate (prune toxic bullets, no LLM)
#   6. Playbook evolves -> next epoch sees updated strategies
#
# NOTE: We use rule-based curate instead of LLM reflect/curate here
# because HF model.generate() is too slow (30 sequential calls = 15min).
# The reward fn already tags bullets, so we just prune the bad ones.
# ===================================================================
print(f"\n[Phase 3] Initializing ACE+TTRL co-evolution ({CFG.GRPO_EPOCHS} epochs)...")

# Initialize shared state
_ace_ttrl_state["playbook_mgr"] = ActivePlaybook(sync_generate)
_ace_ttrl_state["episode_stats"] = []
_ace_ttrl_state["pending_curate"] = []

# Build initial dataset (with initial playbook context)
initial_pb_ctx = _ace_ttrl_state["playbook_mgr"].get_context()
initial_prompts = [format_prompt(p["problem"], initial_pb_ctx) for p in problems]
_ace_ttrl_state["problem_lookup"] = {
    prompt: p for prompt, p in zip(initial_prompts, problems)
}

grpo_trainer_ace_ttrl = GRPOTrainer(
    model=CFG.MODEL_NAME,
    reward_funcs=ace_ttrl_reward_fn,
    args=ace_ttrl_grpo_config,
    train_dataset=Dataset.from_dict({"prompt": initial_prompts}),
    peft_config=peft_config,
    processing_class=tokenizer,
)
# Set model ref for sync_generate fallback (not used in rule-based path)
_sync_model_ref["model"] = grpo_trainer_ace_ttrl.model
print(f"ACE+TTRL GRPOTrainer initialized ({CFG.GRPO_EPOCHS} manual epochs)")

print("Starting ACE+TTRL co-evolution training...")
ace_ttrl_epoch_metrics = []
try:
    for epoch in range(CFG.GRPO_EPOCHS):
        # 1. Rebuild prompts with current playbook context
        playbook_ctx = _ace_ttrl_state["playbook_mgr"].get_context()
        epoch_prompts = [format_prompt(p["problem"], playbook_ctx) for p in problems]

        # 2. Update dataset and problem lookup for this epoch
        grpo_trainer_ace_ttrl.train_dataset = Dataset.from_dict({"prompt": epoch_prompts})
        _ace_ttrl_state["problem_lookup"] = {
            prompt: p for prompt, p in zip(epoch_prompts, problems)
        }
        _ace_ttrl_state["pending_curate"] = []

        # 3. Reset trainer state for clean epoch (preserves optimizer + model)
        reset_trainer_state(grpo_trainer_ace_ttrl)

        # 4. Train 1 epoch (vLLM generates with playbook-injected prompts)
        grpo_trainer_ace_ttrl.train()

        # 5. Rule-based curate: prune toxic bullets, no LLM calls needed.
        # The reward fn already tagged bullets as helpful/harmful.
        rule_based_curate(_ace_ttrl_state["playbook_mgr"],
                         _ace_ttrl_state["pending_curate"])

        pb_size = _ace_ttrl_state["playbook_mgr"].playbook.size
        ace_ttrl_epoch_metrics.append({"epoch": epoch, "pb_size": pb_size})
        print(f"  ACE+TTRL epoch {epoch+1}/{CFG.GRPO_EPOCHS}: pb_size={pb_size}")

    print("ACE+TTRL co-evolution training complete!")
except Exception as e:
    print(f"ACE+TTRL training error: {e}")
    import traceback; traceback.print_exc()

# Evaluate the ACE+TTRL trained model (merge LoRA + vLLM)
result_ace_ttrl = evaluate_trained_model(
    "ace_ttrl", problems, grpo_trainer_ace_ttrl, tokenizer,
    num_episodes=CFG.TRAINED_EVAL_EPISODES,
)
result_ace_ttrl["playbook_snapshots"] = [_ace_ttrl_state["playbook_mgr"].snapshot()]
result_ace_ttrl["ace_ttrl_training_stats"] = _ace_ttrl_state["episode_stats"]
result_ace_ttrl["ace_ttrl_epoch_metrics"] = ace_ttrl_epoch_metrics
all_results["ace_ttrl"] = result_ace_ttrl

with open(os.path.join(CFG.RESULTS_DIR, "ace_ttrl.json"), "w") as f:
    json.dump(result_ace_ttrl, f, indent=2, default=str)
print(f"ACE+TTRL saved. Final acc: {result_ace_ttrl['episodes'][-1]['accuracy']:.1%}")

# Save ACE+TTRL checkpoint and final playbook
grpo_trainer_ace_ttrl.save_model(os.path.join(CFG.CHECKPOINTS_DIR, "ace_ttrl_final"))
with open(os.path.join(CFG.RESULTS_DIR, "ace_ttrl_playbook_final.json"), "w") as f:
    json.dump(_ace_ttrl_state["playbook_mgr"].snapshot(), f, indent=2)

# Clean up
del grpo_trainer_ace_ttrl
_sync_model_ref["model"] = None
torch.cuda.empty_cache()

# ===================================================================
# Save combined results
# ===================================================================
with open(os.path.join(CFG.RESULTS_DIR, "all_results.json"), "w") as f:
    json.dump(all_results, f, indent=2, default=str)

print("\n" + "=" * 60)
print("All conditions complete! Results saved to", CFG.RESULTS_DIR)
print("=" * 60)
for k, v in all_results.items():
    final_acc = v["episodes"][-1]["accuracy"]
    n_eps = len(v["episodes"])
    print(f"  {v['name']:25s}: {final_acc:.1%} ({n_eps} episodes)")

In [ ]:
# ---------------------------------------------------------------------------
# Group D: Analysis & Decision (REQ-6)
# ---------------------------------------------------------------------------
from scipy import stats

# Load results if running analysis cell independently
if "all_results" not in dir() or not all_results:
    with open(os.path.join(CFG.RESULTS_DIR, "all_results.json")) as f:
        all_results = json.load(f)

# ---------------------------------------------------------------------------
# Bootstrap CI function
# ---------------------------------------------------------------------------

def bootstrap_ci(data, n_bootstrap=10000, ci=0.95, stat_fn=np.mean):
    """Bootstrap confidence interval for a statistic.

    Returns (point_estimate, lower, upper) where point_estimate is the
    sample statistic (not the bootstrap mean).
    """
    rng = np.random.default_rng(42)
    boot_stats = []
    data = np.array(data)
    for _ in range(n_bootstrap):
        sample = rng.choice(data, size=len(data), replace=True)
        boot_stats.append(stat_fn(sample))
    boot_stats = np.array(boot_stats)
    lower = np.percentile(boot_stats, (1 - ci) / 2 * 100)
    upper = np.percentile(boot_stats, (1 + ci) / 2 * 100)
    return float(stat_fn(data)), float(lower), float(upper)

# ---------------------------------------------------------------------------
# Extract per-problem binary outcomes for the LAST episode of each condition
# (Each condition may have a different number of episodes)
# ---------------------------------------------------------------------------

conditions = ["baseline", "ace_only", "ttrl_only", "ace_ttrl"]
condition_labels = {
    "baseline": "Baseline",
    "ace_only": "ACE-only",
    "ttrl_only": "TTRL-only",
    "ace_ttrl": "ACE+TTRL",
}

binary_outcomes = {}
for cond in conditions:
    episodes = all_results[cond]["episodes"]
    last_ep = len(episodes) - 1
    details = episodes[last_ep]["details"]
    binary_outcomes[cond] = [d["correct"] for d in details]

# ---------------------------------------------------------------------------
# McNemar's test for paired comparisons (exact binomial)
# ---------------------------------------------------------------------------

def mcnemar_test(outcomes_a, outcomes_b):
    """McNemar's test with exact binomial — appropriate for small N.

    Chi-squared approximation is unreliable when the number of discordant
    pairs is fewer than ~25. With N=30 problems, exact binomial is correct.
    """
    a = np.array(outcomes_a, dtype=bool)
    b = np.array(outcomes_b, dtype=bool)
    # Discordant pairs
    n01 = int(np.sum(~a & b))  # a wrong, b right
    n10 = int(np.sum(a & ~b))  # a right, b wrong
    n = n01 + n10
    if n == 0:
        return 1.0, n01, n10
    # Exact binomial test (two-sided): H0 is that discordant pairs are
    # equally likely to favor either direction (p=0.5)
    p_value = stats.binomtest(n01, n, 0.5).pvalue
    return float(p_value), n01, n10

# Key comparisons
print("=" * 60)
print("Statistical Analysis")
print("=" * 60)

comparisons = [
    ("ace_ttrl", "ttrl_only", "ACE+TTRL vs TTRL-only"),
    ("ace_ttrl", "ace_only", "ACE+TTRL vs ACE-only"),
    ("ace_ttrl", "baseline", "ACE+TTRL vs Baseline"),
    ("ttrl_only", "baseline", "TTRL-only vs Baseline"),
    ("ace_only", "baseline", "ACE-only vs Baseline"),
]

print(f"\n{'Comparison':35s} {'p-value':>10s} {'n01':>5s} {'n10':>5s} {'Sig':>5s}")
print("-" * 65)
for cond_a, cond_b, label in comparisons:
    p_val, n01, n10 = mcnemar_test(binary_outcomes[cond_a], binary_outcomes[cond_b])
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    print(f"{label:35s} {p_val:10.4f} {n01:5d} {n10:5d} {sig:>5s}")

# ---------------------------------------------------------------------------
# Bootstrap CIs on final accuracy
# ---------------------------------------------------------------------------

print(f"\n{'Condition':15s} {'Accuracy':>10s} {'95% CI':>20s} {'Episodes':>10s}")
print("-" * 60)
for cond in conditions:
    outcomes = binary_outcomes[cond]
    mean, lo, hi = bootstrap_ci(outcomes, n_bootstrap=10000)
    n_eps = len(all_results[cond]["episodes"])
    print(f"{condition_labels[cond]:15s} {mean:10.1%} [{lo:.1%}, {hi:.1%}] {n_eps:>10d}")

# Key difference: ACE+TTRL minus TTRL-only
ace_ttrl_outcomes = np.array(binary_outcomes["ace_ttrl"], dtype=float)
ttrl_outcomes = np.array(binary_outcomes["ttrl_only"], dtype=float)
diff = ace_ttrl_outcomes - ttrl_outcomes
diff_mean, diff_lo, diff_hi = bootstrap_ci(diff, n_bootstrap=10000)
print(f"\nACE+TTRL - TTRL-only: {diff_mean:+.1%} [{diff_lo:+.1%}, {diff_hi:+.1%}]")

# Note on statistical power
print(f"\nNote: With N=30 problems, this experiment has power to detect ~15-20%")
print(f"accuracy differences at alpha=0.05. Smaller effects require more problems.")
print(f"This is a PoC with {CFG.GRPO_EPOCHS} GRPO epochs (paper uses 60).")

# ---------------------------------------------------------------------------
# Plots: 2x2 grid
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {"baseline": "#888888", "ace_only": "#2196F3", "ttrl_only": "#FF9800", "ace_ttrl": "#4CAF50"}

# (a) Accuracy over episodes (where applicable)
ax = axes[0, 0]
for cond in conditions:
    accs = [ep["accuracy"] for ep in all_results[cond]["episodes"]]
    if len(accs) > 1:
        ax.plot(range(1, len(accs) + 1), accs, label=condition_labels[cond],
                color=colors[cond], linewidth=2)
    else:
        # Single episode: show as horizontal line
        ax.axhline(y=accs[0], label=f"{condition_labels[cond]} ({accs[0]:.1%})",
                    color=colors[cond], linewidth=2, linestyle="--")
ax.set_xlabel("Episode")
ax.set_ylabel("Accuracy")
ax.set_title("(a) Accuracy Over Episodes")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# (b) Playbook size over time (ACE conditions only)
ax = axes[0, 1]
for cond in ["ace_only", "ace_ttrl"]:
    snapshots = all_results[cond].get("playbook_snapshots", [])
    sizes = []
    for snap in snapshots:
        if snap.get("type") == "null":
            sizes.append(0)
        elif "playbook" in snap:
            sizes.append(len(snap["playbook"].get("bullets", [])))
        else:
            sizes.append(0)
    if sizes:
        ax.plot(range(1, len(sizes) + 1), sizes, label=condition_labels[cond],
                color=colors[cond], linewidth=2)
# Also plot ACE+TTRL epoch metrics if available
if "ace_ttrl_epoch_metrics" in all_results.get("ace_ttrl", {}):
    metrics = all_results["ace_ttrl"]["ace_ttrl_epoch_metrics"]
    if metrics:
        sizes = [m["pb_size"] for m in metrics]
        ax.plot(range(1, len(sizes) + 1), sizes, label="ACE+TTRL (training)",
                color=colors["ace_ttrl"], linewidth=2, linestyle="--")
ax.set_xlabel("Episode / Epoch")
ax.set_ylabel("Playbook Size (bullets)")
ax.set_title("(b) Playbook Size Over Time")
ax.legend()
ax.grid(True, alpha=0.3)

# (c) Reward accuracy: how often does majority vote match ground truth?
ax = axes[1, 0]
for cond in ["ace_only", "ttrl_only", "ace_ttrl"]:
    episodes = all_results[cond]["episodes"]
    reward_accs = [ep["accuracy"] for ep in episodes]
    if len(reward_accs) > 1:
        ax.plot(range(1, len(reward_accs) + 1), reward_accs, label=condition_labels[cond],
                color=colors[cond], linewidth=2)
    else:
        ax.axhline(y=reward_accs[0], label=condition_labels[cond],
                    color=colors[cond], linewidth=2, linestyle="--")
ax.set_xlabel("Episode")
ax.set_ylabel("Majority Vote Accuracy")
ax.set_title("(c) Reward Signal Quality Over Episodes")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# (d) Final accuracy bar chart with confidence intervals
ax = axes[1, 1]
x_pos = np.arange(len(conditions))
means = []
ci_lower = []
ci_upper = []
bar_colors = [colors[c] for c in conditions]

for cond in conditions:
    outcomes = binary_outcomes[cond]
    mean, lo, hi = bootstrap_ci(outcomes, n_bootstrap=10000)
    means.append(mean)
    ci_lower.append(mean - lo)
    ci_upper.append(hi - mean)

bars = ax.bar(x_pos, means, color=bar_colors, alpha=0.8, edgecolor="black")
ax.errorbar(x_pos, means, yerr=[ci_lower, ci_upper], fmt="none",
            ecolor="black", capsize=5, linewidth=1.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([condition_labels[c] for c in conditions], rotation=15)
ax.set_ylabel("Accuracy")
ax.set_title("(d) Final Accuracy (Last Episode) with 95% CI")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis="y")

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.02,
            f"{mean:.1%}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(CFG.RESULTS_DIR, "analysis_plots.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Plots saved to {os.path.join(CFG.RESULTS_DIR, 'analysis_plots.png')}")

# ---------------------------------------------------------------------------
# Decision verdict
# ---------------------------------------------------------------------------

print("\n" + "=" * 60)
print("DECISION VERDICT")
print("=" * 60)

ace_ttrl_acc = all_results["ace_ttrl"]["episodes"][-1]["accuracy"]
ttrl_acc = all_results["ttrl_only"]["episodes"][-1]["accuracy"]
ace_acc = all_results["ace_only"]["episodes"][-1]["accuracy"]
base_acc = all_results["baseline"]["episodes"][-1]["accuracy"]

gap = ace_ttrl_acc - ttrl_acc
print(f"  Baseline:     {base_acc:.1%}")
print(f"  ACE-only:      {ace_acc:.1%}")
print(f"  TTRL-only:    {ttrl_acc:.1%}")
print(f"  ACE+TTRL:      {ace_ttrl_acc:.1%}")
print(f"  Gap (ACE+TTRL - TTRL-only): {gap:+.1%}")
print()

if gap > 0.10:
    verdict = "CO-EVOLUTION WINS"
    next_step = "Co-evolution produces synergistic improvement. Scale up: more epochs (60), larger model, add strong-model verifier."
elif gap > 0.05:
    verdict = "MARGINAL SYNERGY"
    next_step = "Small improvement from co-evolution. Try: more epochs, verifier-filtered reward, archetype-anchored curriculum."
elif gap > -0.05:
    verdict = "GRPO SUBSUMES PLAYBOOK"
    next_step = "Weight updates capture what the playbook provides. Focus on TTRL improvements: more data, curriculum, reward engineering."
else:
    verdict = "PLAYBOOK INTERFERES WITH RL"
    next_step = "Playbook operations hurt training. Investigate: reward noise from reflect/curate, or playbook poisoning under RL."

print(f"  VERDICT: {verdict}")
print(f"  NEXT:    {next_step}")

# Save analysis
analysis = {
    "final_accuracies": {cond: all_results[cond]["episodes"][-1]["accuracy"] for cond in conditions},
    "gap_acettrl_minus_ttrl": gap,
    "verdict": verdict,
    "next_step": next_step,
    "grpo_epochs": CFG.GRPO_EPOCHS,
    "note": f"PoC scale ({CFG.GRPO_EPOCHS} epochs vs 60 in TTRL paper)",
}
with open(os.path.join(CFG.RESULTS_DIR, "analysis.json"), "w") as f:
    json.dump(analysis, f, indent=2)
print(f"\nAnalysis saved to {os.path.join(CFG.RESULTS_DIR, 'analysis.json')}")